# Credit Risk PD Model V1

This notebook builds an entry-level probability of default (PD) model using the public German Credit dataset from OpenML. The goal is to practise the modelling discipline used in credit risk: define a binary default target, train interpretable and non-linear models, evaluate discrimination, check calibration, and document limitations.

This is a learning prototype, not a regulatory credit model.

## 1. Imports and Configuration

In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, confusion_matrix, roc_auc_score, RocCurveDisplay
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42

## 2. Load Credit Dataset

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/selva86/datasets/master/GermanCredit.csv'
df = pd.read_csv(DATA_URL)
df.columns = [c.lower().replace(' ', '_').replace('-', '_') for c in df.columns]

# The CSV uses credit_risk where 0 is bad/risky and 1 is good. Treat risky borrowers as default = 1.
target_col = 'credit_risk'
df['default'] = 1 - pd.to_numeric(df[target_col], errors='coerce').astype(int)
df = df.drop(columns=[target_col])
df.to_csv(RAW_DIR / 'german_credit_raw.csv', index=False)

print(df.shape)
display(df.head())
display(df['default'].value_counts(normalize=True).rename('default_rate'))

## 3. Train/Test Split and Preprocessing

In [ ]:
X = df.drop(columns=['default'])
y = df['default']

numeric_cols = X.select_dtypes(include=['number']).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_SEED, stratify=y
)

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocess = ColumnTransformer([
    ('num', numeric_pipe, numeric_cols),
    ('cat', categorical_pipe, categorical_cols),
])

print('Numeric columns:', numeric_cols)
print('Categorical columns:', len(categorical_cols))

## 4. Model Training

In [ ]:
models = {
    'logistic_regression': Pipeline([
        ('preprocess', preprocess),
        ('model', LogisticRegression(max_iter=2000, class_weight='balanced')),
    ]),
    'random_forest': Pipeline([
        ('preprocess', preprocess),
        ('model', RandomForestClassifier(
            n_estimators=300, max_depth=5, min_samples_leaf=10,
            class_weight='balanced_subsample', random_state=RANDOM_SEED
        )),
    ]),
}

results = []
predictions = pd.DataFrame({'actual_default': y_test.values}, index=y_test.index)

for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    predictions[f'{name}_pd'] = proba
    results.append({
        'model': name,
        'roc_auc': roc_auc_score(y_test, proba),
        'brier_score': brier_score_loss(y_test, proba),
        'accuracy_at_0.50': accuracy_score(y_test, pred),
    })

results_df = pd.DataFrame(results).sort_values('roc_auc', ascending=False)
display(results_df)
predictions.to_csv(PROCESSED_DIR / 'credit_risk_pd_predictions_v1.csv', index=False)

## 5. Evaluation and Calibration

In [ ]:
best_name = results_df.iloc[0]['model']
best_pd_col = f'{best_name}_pd'
best_pd = predictions[best_pd_col]

RocCurveDisplay.from_predictions(y_test, best_pd)
plt.title(f'ROC Curve: {best_name}')
plt.show()

cm = confusion_matrix(y_test, (best_pd >= 0.5).astype(int))
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['non-default', 'default'], yticklabels=['non-default', 'default'])
plt.title('Confusion Matrix at 50% PD Cutoff')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

calibration = predictions.copy()
calibration['pd_bucket'] = pd.qcut(calibration[best_pd_col].rank(method='first'), 5, labels=False) + 1
bucket_summary = calibration.groupby('pd_bucket').agg(
    avg_predicted_pd=(best_pd_col, 'mean'),
    realized_default_rate=('actual_default', 'mean'),
    observations=('actual_default', 'size'),
).reset_index()
display(bucket_summary)

plt.figure(figsize=(6, 4))
sns.lineplot(data=bucket_summary, x='avg_predicted_pd', y='realized_default_rate', marker='o')
plt.plot([0, 1], [0, 1], color='black', linestyle='--')
plt.title('PD Calibration by Score Bucket')
plt.xlabel('Average predicted PD')
plt.ylabel('Realized default rate')
plt.show()

## 6. Risk Modelling Notes

- PD is a probability, not only a class label. Calibration matters.
- ROC-AUC measures discrimination: whether higher-risk borrowers are ranked above lower-risk borrowers.
- Brier score measures probability quality.
- This public dataset is small and not point-in-time. A production model would require governance, monitoring, explainability, validation, and regulatory controls.